Criando um sistema RAG simples

fonte: https://huggingface.co/blog/ngxson/make-your-own-rag


1. Indexação -> 2. Recuperação

In [7]:
# Carregando o dataset

with open('dataset/cat-facts.txt', 'r') as file:
    dataset = file.readlines()
    print(f'Loaded{len(dataset)} entries')

Loaded150 entries


In [8]:
# Implementando o banco de vetores

# usamos o vetor de embedding para converter cada chunk em um vetor de embedding, depois adiciona na lista

import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'


VECTOR_DB = []

def add_chunk_to_database(chunk):
    embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
    VECTOR_DB.append((chunk, embedding))
    
# para facilitar, nesse exemplo vamos considerar cada linha como um chunk
for i, chunk in enumerate(dataset):
    add_chunk_to_database(chunk)
    print(f'[{i+1}]/{len(dataset)} | {chunk} adicionado ao banco de embeddings')
    

ConnectionError: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download

In [ ]:
# Implementando função de Recuperação
from sklearn.metrics.pairwise import cosine_similarity

# encapsulando 
def similaridade_cosseno(a, b):
    return cosine_similarity([a], [b])[0][0]
    
def recuperacao(query, top_n=3):
    query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
    # guardando lista temporaria de pares (chunk, similarity)
    similarities = []
    for chunk, embedding in VECTOR_DB:
        similarity = similaridade_cosseno(query_embedding, embedding)
        similarities.append((chunk, similarity))
    
    # ordenando por similaridade em ordem decrescente, maiore similaridades significam chunks mais relevantes
    similarities.sort(key=lambda x: x[1], reverse=True)
    # returnando o top N chunks mais relevantes
    return similarities[:top_n]

In [ ]:
input_query = input('Faça uma pergunta: ')

retrieved_knowledge = recuperacao(input_query)
print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{'\n'.join([f' - {chunk}' for chunk, similarity in retrieved_knowledge])}
'''

In [ ]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)
# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)
